# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.** I chose this lane because a content team has limited review time and needs to decide which pages deserve attention first. The goal is an explainable ranked review queue, not an automatic instruction to edit a page and not a model for its own sake. I will start with a transparent rule baseline and test whether observed, pre-decision search, content, and engagement signals improve prioritization under honest validation.

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Starter CSV not found. Run this notebook from the repo root or work/notebooks.")

df = pd.read_csv(data_path)
required_columns = {"content_id", "impressions_90d", "trend_direction"}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert df["content_id"].is_unique, "Expected one row per content item."
print("Starter data loaded; the one-row-per-page grain check passed.")

Starter data loaded; the one-row-per-page grain check passed.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough prior search visibility to support a reliable comparison, which pages should a content strategist review first because they show evidence of a meaningful next-window visibility risk or opportunity?

- **Unit of analysis:** one pseudonymized content item (page) at one decision date.
- **Decision:** how to allocate the next 20 page-review slots.
- **Output:** a ranked top-20 review queue with a score, confidence label, and observable reason codes.
- **Action:** a content strategist reviews the evidence and then chooses whether to protect, refresh, expand, consolidate, prune, or monitor the page. The score does not make that choice automatically.
- **Cost of a wrong call:** a false positive wastes scarce review or editing time and an unnecessary edit could damage a page that is performing well. A false negative can delay attention to a valuable page that is losing visibility. Because false positives can trigger costly work, the provisional primary metric is **precision@20**; I will also report recall so missed candidates stay visible.

### Provisional task definition

For a content strategist deciding which 20 pages to review next, I will build an explainable ranking from pseudonymized measurements available before the decision date. For later warehouse work, the provisional outcome will be a material change in search visibility measured in a separate future window, with a minimum-volume rule to reduce noise. Performance will be measured with precision@20 on held-out clients and later time windows. A wrong call costs review time or a missed opportunity. A plain rule is the baseline; ML earns a place only if several interacting signals produce a stable, useful improvement over that rule. Any result will be framed as observed, directional decision-support.

## 3. Quick look at the data (2-3 real numbers)

The starter file is a trailing-90-day snapshot, so these numbers show the size of the prioritization problem; they do **not** test future performance or the effect of editing a page. I use 500 impressions as a provisional volume guard for this first look. The current `trend_direction` value is a descriptive, rule-derived signal here—not a future target and never a feature for reproducing that same label.

In [2]:
visible = df["impressions_90d"].ge(500)
down_now = df["trend_direction"].eq("down")

summary = pd.DataFrame(
    {
        "Measured quantity": [
            "Pages in the starter slice",
            "Pages with at least 500 trailing-90-day impressions",
            "Pages with at least 500 impressions and current direction = down",
        ],
        "Page count": [len(df), int(visible.sum()), int((visible & down_now).sum())],
    }
)
summary["Share of slice"] = summary["Page count"] / len(df)
summary.style.format({"Page count": "{:,}", "Share of slice": "{:.1%}"})

,Measured quantity,Page count,Share of slice
0,Pages in the starter slice,"30,000",100.0%
1,Pages with at least 500 trailing-90-day impressions,"16,726",55.8%
2,Pages with at least 500 impressions and current direction = down,"9,961",33.2%


## 4. Careful words: what I can and can't claim

The quick look **measures** that the starter slice contains many visible pages with current downward movement. This is enough to motivate a ranking problem, but it is not evidence that a refresh would help. Later, if a method performs well on a pre-declared, future observed outcome using held-out clients and time windows, I can say it provides **directional decision-support** for which pages to review first within this release.

I cannot claim that a recommendation causes traffic recovery, that a low CTR proves bad metadata, that a model has learned or predicts Google's algorithm, or that results automatically generalize to other clients or dates. The starter `trend_direction` and `trend_pct` fields are derived from the same current comparison window, so they cannot be used as features when current decline is the label. A future model must keep feature and outcome windows separate and still leave the final action to a human reviewer.

### Why this is not just “train a model”

The project succeeds only if it improves a real review decision. I will define the outcome and metric first, audit the signals, build a simple rule baseline, compare any model against that baseline, inspect the top 20 by hand, and attach reason codes and limits to the queue. If a model does not improve precision@20 consistently enough to justify its complexity, the transparent baseline remains the better result.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.